# ML: подготовка данных для прогноза нагрузки на следующие сутки

Первый шаг этапа 2 из [бизнес-задачи](../BUSINESS_TASK.md): получить общий ряд с шагом 15 минут, проверить его и разделить по времени. Модели и оценки точности здесь ещё не рассчитываются.

**Результат:** `total_load`, `train`, `validation`, `test` и локальный CSV в `data/ml/total_load_15min.csv`. Исходный файл и EDA-ноутбук не изменяются.

## 1. Параметры прогноза

Рабочий горизонт — **96 интервалов (сутки)** с шагом 15 минут. Целевая величина — совокупная средняя мощность интервала, кВт; энергия интервала равна мощности × 0,25 часа.

Источник: [UCI ElectricityLoadDiagrams20112014](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014), файл `data/LD2011_2014.txt`. Границы валидации и теста заданы ниже заранее.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Находим корень проекта для импорта общих модулей.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_load.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Не найден корень проекта ElectricityLoad.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_load import get_data, get_file_sha256
from src.data_validation import get_dataset_overview, validate_ml_source_data
from src.ml_preparation import (
    prepare_total_load, split_load_by_time, get_split_summary, save_prepared_load,
)

INTERVAL_MINUTES = 15
HORIZON = 96
VALIDATION_START = pd.Timestamp("2014-01-01")
TEST_START = pd.Timestamp("2014-07-01")
DATA_PATH = project_root / "data" / "LD2011_2014.txt"
OUTPUT_PATH = project_root / "data" / "ml" / "total_load_15min.csv"

## 2. Загрузка оригинальных данных

Читаем все клиентские столбцы: разделитель `;`, десятичная запятая, первая колонка — временная метка конца интервала. Размер исходного файла около 711 МБ; для загрузки и проверок нужно несколько гигабайт памяти. Контрольная сумма фиксирует именно использованный файл, но не заменяет проверку происхождения.

In [2]:
raw_data = get_data(DATA_PATH)
raw_data.index.name = "interval_end"
source_sha256 = get_file_sha256(DATA_PATH)
dataset_overview = get_dataset_overview(raw_data)
print(f"SHA-256 исходного файла: {source_sha256}")

Датасет загружен из: /workspace/ElectricityLoad/data/LD2011_2014.txt


ОБЩАЯ ИНФОРМАЦИЯ О ДАТАСЕТЕ
---------------------------------------------
Строки (временные отметки): 140,256
Столбцы (клиенты / точки учёта): 370
Начало наблюдений: 2011-01-01 00:15:00
Конец наблюдений:  2015-01-01 00:00:00
Первые 10 клиентов: ['MT_001', 'MT_002', 'MT_003', 'MT_004', 'MT_005', 'MT_006', 'MT_007', 'MT_008', 'MT_009', 'MT_010']

Количество столбцов по типам данных:
float64    370
Name: count, dtype: int64
SHA-256 исходного файла: d51565f2cb5a6b768d06ba1bbd3c084c6e2f3aab07f00c6f2dcb80e90175124b


## 3. Проверки качества

Проверки останавливают выполнение при нарушении структуры: пропуски, бесконечности, отрицательные значения, дубликаты или разрывы сетки. Мы не исправляем эти проблемы автоматически.

**Нули сохраняем:** они могут отражать начало истории клиента или особенности перевода часов. Метки остаются в исходном локальном времени без назначения UTC: набор имеет специальное представление переходов часов. Разный период активности клиентов означает меняющийся состав общей нагрузки — это нужно учитывать при оценке будущего прогноза.

In [3]:
quality = validate_ml_source_data(
    raw_data, interval_minutes=INTERVAL_MINUTES, min_intervals=HORIZON,
)
display(quality)
print("Проверки исходных данных пройдены.")

ПРОВЕРКА ВРЕМЕННОГО ИНДЕКСА И ДАННЫХ
--------------------------------------------------
Индекс расположен по возрастанию: True
Повторяющиеся временные отметки: 0
Явные пропуски NaN: 0

Наиболее частые интервалы между наблюдениями:
15 минут — 140,255 раз


missing_values                   0
duplicate_timestamps             0
zero_values               10457342
negative_values                  0
nonfinite_values                 0
duplicate_client_names           0
all_zero_clients                 0
all_zero_timestamps              0
Name: quality, dtype: int64

Проверки исходных данных пройдены.


## 4. Общий ряд нагрузки

Суммируем мощность **всех** клиентов, без исключения клиентов по правилам кластеризации. Переносим метку конца интервала на его начало: `00:15` становится `00:00`. Так каждый день состоит из 96 интервалов.

`total_power_kw` — будущая целевая величина; `total_energy_kwh` — та же нагрузка в единицах энергии, а не независимый прогнозный признак.

In [4]:
total_load = prepare_total_load(raw_data, interval_minutes=INTERVAL_MINUTES)
display(total_load.head())

,total_power_kw,total_energy_kwh
timestamp,,
2011-01-01 00:00:00,68513.115338,17128.278835
2011-01-01 00:15:00,69180.304361,17295.076090
2011-01-01 00:30:00,69364.850573,17341.212643
2011-01-01 00:45:00,68350.480660,17087.620165
2011-01-01 01:00:00,66166.874303,16541.718576


## 5. Разделение только по времени

- Обучение: до 1 января 2014 года.
- Валидация: январь–июнь 2014 года.
- Тест: июль–декабрь 2014 года.

Границы заданы заранее и проходят по началу суток. Случайное перемешивание не используем. Гиперпараметры будем выбирать на валидации; тест оставим для итоговой оценки. Для суточного прогноза будущие признаки должны быть известны в момент начала суток: нельзя подставлять фактическую нагрузку уже прогнозируемых интервалов.

In [5]:
splits = split_load_by_time(
    total_load, VALIDATION_START, TEST_START, interval_minutes=INTERVAL_MINUTES,
)
train = splits["train"]
validation = splits["validation"]
test = splits["test"]

split_summary = get_split_summary(splits, interval_minutes=INTERVAL_MINUTES)
display(split_summary)

,start,end,intervals,days
split,,,,
train,2011-01-01,2013-12-31 23:45:00,105216,1096
validation,2014-01-01,2014-06-30 23:45:00,17376,181
test,2014-07-01,2014-12-31 23:45:00,17664,184


## 6. Сохранение подготовленных данных

Экспортируем общий ряд с колонкой `split`. Файл создаётся в игнорируемой Git папке `data/ml/`; при повторном выполнении обновляется только этот производный CSV. Он и исходный датасет не отправляются в GitHub. Ноутбук содержит код и компактные результаты проверок.

In [6]:
prepared_data = save_prepared_load(splits, OUTPUT_PATH)
print(f"Сохранено: data/ml/{OUTPUT_PATH.name}")
print(f"Интервалов: {len(prepared_data):,}; клиентов в исходной сумме: {len(raw_data.columns)}")
print("Проверки агрегации, полных суток, разделения и повторного чтения CSV пройдены.")

Сохранено: data/ml/total_load_15min.csv
Интервалов: 140,256; клиентов в исходной сумме: 370
Проверки агрегации, полных суток, разделения и повторного чтения CSV пройдены.


## Следующий шаг

На одинаковых суточных точках старта сравнить три базовых прогноза на 96 интервалов: повтор последнего известного значения, профиль предыдущих суток и профиль недели назад. Рассчитать MAE, RMSE и WAPE на валидации, затем добавить одну простую ML-модель.

На этом этапе подготовлены данные и временные выборки. Прогнозы ещё не построены; точность и полезность модели не оценивались. CSV не содержит лагов, обученных преобразований или будущих целевых векторов.